1. Title

Employee Management System Using PostgreSQL and Python

2. Aim

To develop a Python-based employee management system using a PostgreSQL relational database to create, retrieve, and manage employees, departments, and roles while demonstrating basic CRUD operations.

3. Theory

A relational database stores data in tables consisting of rows and columns. PostgreSQL is a relational database management system that uses SQL (Structured Query Language) to create, retrieve, update, and delete data.

In this experiment, a Python-based system is developed to manage employees, departments, and roles using PostgreSQL. The system performs basic CRUD operations: Create, Read, Update, and Delete. Python is used to connect with the PostgreSQL database and execute SQL queries for managing the employee-related data.


Install PostgreSQL

In [ ]:
!apt-get update -qq
!apt-get install -y postgresql postgresql-contrib -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package logrotate.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../00-logrotate_3.19.0-1ubuntu1.1_amd64.deb ...
Unpacking logrotate (3.19.0-1ubuntu1.1) ...
Selecting previously unselected package netbase.
Preparing to unpack .../01-netbase_6.3_all.deb ...
Unpacking netbase (6.3) ...
Selecting previously unselected package libcommon-sense-perl:amd64.
Preparing to unpack .../02-libcommon-sense-perl_3.75-2build1_amd64.deb ...
Unpacking libcommon-sense-perl:amd64 (3.75-2build1) ...
Selecting previously unselected package libjson-perl.
Preparing to unpack .../03-libjson-perl_4.04000-1_all.deb ...
Unpacking libjson-perl (4.04000-1) ...
Selecting previously unselected package libtypes-serialiser-perl

Start PostgreSQL

In [ ]:
!service postgresql start

 * Starting PostgreSQL 14 database server
   ...done.


Install Python PostgreSQL connector

In [ ]:
!pip install psycopg2-binary -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 56.7 MB/s eta 0:00:00


Import required library

In [ ]:
import psycopg2

In [ ]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"

ALTER ROLE


In [ ]:
!service postgresql start

 * Starting PostgreSQL 14 database server
   ...done.


Connect Python to PostgreSQL

In [ ]:
import psycopg2

connection = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="postgres"
)

cursor = connection.cursor()

print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


Create the database tables

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS departments (
    department_id SERIAL PRIMARY KEY,
    department_name VARCHAR(100) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS roles (
    role_id SERIAL PRIMARY KEY,
    role_name VARCHAR(100) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    employee_id SERIAL PRIMARY KEY,
    employee_name VARCHAR(100) NOT NULL,
    email VARCHAR(100),
    department_id INT REFERENCES departments(department_id),
    role_id INT REFERENCES roles(role_id)
);
""")

connection.commit()

print("Tables created successfully!")

Tables created successfully!


Create department records

In [ ]:
cursor.execute("""
INSERT INTO departments (department_name)
VALUES
('Human Resources'),
('Finance'),
('Technology')
RETURNING department_id;
""")

department_ids = cursor.fetchall()

connection.commit()

print("Departments created successfully!")

Departments created successfully!


Create role records

In [ ]:
cursor.execute("""
INSERT INTO roles (role_name)
VALUES
('Manager'),
('Developer'),
('Analyst')
RETURNING role_id;
""")

role_ids = cursor.fetchall()

connection.commit()

print("Roles created successfully!")

Roles created successfully!


Create employee records

In [ ]:
cursor.execute("""
INSERT INTO employees
(employee_name, email, department_id, role_id)
VALUES
('Rahul Sharma', 'rahul@example.com', 3, 2),
('Priya Patil', 'priya@example.com', 1, 1),
('Amit Joshi', 'amit@example.com', 2, 3)
""")

connection.commit()

print("Employees created successfully!")

Employees created successfully!


Retrieve employees

In [ ]:
cursor.execute("""
SELECT * FROM employees;
""")

employees = cursor.fetchall()

for employee in employees:
    print(employee)

(1, 'Rahul Sharma', 'rahul@example.com', 3, 2)
(2, 'Priya Patil', 'priya@example.com', 1, 1)
(3, 'Amit Joshi', 'amit@example.com', 2, 3)


Retrieve employee information with department and role

In [ ]:
cursor.execute("""
SELECT
    e.employee_id,
    e.employee_name,
    e.email,
    d.department_name,
    r.role_name
FROM employees e
JOIN departments d
    ON e.department_id = d.department_id
JOIN roles r
    ON e.role_id = r.role_id;
""")

records = cursor.fetchall()

for record in records:
    print(record)

(1, 'Rahul Sharma', 'rahul@example.com', 'Technology', 'Developer')
(2, 'Priya Patil', 'priya@example.com', 'Human Resources', 'Manager')
(3, 'Amit Joshi', 'amit@example.com', 'Finance', 'Analyst')


Update an employee

In [ ]:
cursor.execute("""
UPDATE employees
SET role_id = 1
WHERE employee_name = 'Rahul Sharma';
""")

connection.commit()

print("Employee record updated successfully!")

Employee record updated successfully!


Verify the update

In [ ]:
cursor.execute("""
SELECT
    e.employee_id,
    e.employee_name,
    e.email,
    d.department_name,
    r.role_name
FROM employees e
JOIN departments d
    ON e.department_id = d.department_id
JOIN roles r
    ON e.role_id = r.role_id;
""")

records = cursor.fetchall()

for record in records:
    print(record)

(2, 'Priya Patil', 'priya@example.com', 'Human Resources', 'Manager')
(3, 'Amit Joshi', 'amit@example.com', 'Finance', 'Analyst')
(1, 'Rahul Sharma', 'rahul@example.com', 'Technology', 'Manager')


Delete an employee

In [ ]:
cursor.execute("""
DELETE FROM employees
WHERE employee_name = 'Amit Joshi';
""")

connection.commit()

print("Employee record deleted successfully!")

Employee record deleted successfully!


Verify deletion

In [ ]:
cursor.execute("""
SELECT * FROM employees;
""")

employees = cursor.fetchall()

for employee in employees:
    print(employee)

(2, 'Priya Patil', 'priya@example.com', 1, 1)
(1, 'Rahul Sharma', 'rahul@example.com', 3, 1)


Display final employee information

In [ ]:
cursor.execute("""
SELECT
    e.employee_id,
    e.employee_name,
    e.email,
    d.department_name,
    r.role_name
FROM employees e
JOIN departments d
    ON e.department_id = d.department_id
JOIN roles r
    ON e.role_id = r.role_id;
""")

records = cursor.fetchall()

print("Final Employee Records")
print("-" * 70)

for record in records:
    print(record)

Final Employee Records
----------------------------------------------------------------------
(2, 'Priya Patil', 'priya@example.com', 'Human Resources', 'Manager')
(1, 'Rahul Sharma', 'rahul@example.com', 'Technology', 'Manager')


Close the connection

In [ ]:
cursor.close()
connection.close()

print("PostgreSQL connection closed.")

PostgreSQL connection closed.
